# LendInsight — Notebook 2: EDA Visualizations
**Purpose**: Uncover business patterns through 8 targeted visualizations.  
Each chart answers a specific stakeholder question and maps to a Power BI dashboard page.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.family": "sans-serif"})

CLEAN_PATH = r"C:\data_analyst\LendInsight\01_data\clean\credit_risk_cleaned.csv"
df = pd.read_csv(CLEAN_PATH)
print(f"Dataset: {df.shape[0]:,} rows loaded.")


## Chart 1 — Default Rate by Loan Grade
**Business Question**: Which grade bands are failing most?

In [ ]:

grade_df = df.groupby("loan_grade").agg(
    total=("default_flag","count"),
    defaults=("default_flag","sum")
).reset_index()
grade_df["default_rate"] = grade_df["defaults"] / grade_df["total"] * 100

colors = ["#1A5276" if r < 15 else "#E67E22" if r < 30 else "#C0392B"
          for r in grade_df["default_rate"]]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(grade_df["loan_grade"], grade_df["default_rate"], color=colors, edgecolor="white", width=0.6)
ax.set_title("Default Rate by Loan Grade", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Loan Grade (A = Lowest Risk → G = Highest Risk)", fontsize=11)
ax.set_ylabel("Default Rate (%)", fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, grade_df["default_rate"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f"{val:.1f}%", ha="center", fontweight="bold", fontsize=10)
ax.axhline(y=grade_df["default_rate"].mean(), color="red", linestyle="--", alpha=0.6, label="Portfolio Avg")
ax.legend()
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_05_default_by_grade.png", bbox_inches="tight")
plt.show()
print("Key Finding: Grade G default rate is significantly higher than portfolio average.")


## Chart 2 — Default Rate by Loan Intent
**Business Question**: Which loan purpose has the highest failure rate?

In [ ]:

intent_df = df.groupby("loan_intent").agg(
    total=("default_flag","count"),
    defaults=("default_flag","sum")
).reset_index()
intent_df["default_rate"] = intent_df["defaults"] / intent_df["total"] * 100
intent_df = intent_df.sort_values("default_rate", ascending=True)

colors = ["#C0392B" if r > 25 else "#E67E22" if r > 18 else "#2E86C1"
          for r in intent_df["default_rate"]]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(intent_df["loan_intent"], intent_df["default_rate"], color=colors, edgecolor="white")
ax.set_title("Default Rate by Loan Intent", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Default Rate (%)", fontsize=11)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, intent_df["default_rate"]):
    ax.text(val + 0.3, bar.get_y()+bar.get_height()/2,
            f"{val:.1f}%", va="center", fontweight="bold", fontsize=10)
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_06_default_by_intent.png", bbox_inches="tight")
plt.show()


## Chart 3 — Income Distribution: Defaulters vs Non-Defaulters
**Business Question**: Does income level predict default?

In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))
repaid   = df[df["default_flag"]==0]["person_income"]
defaulted = df[df["default_flag"]==1]["person_income"]

ax.hist(repaid,    bins=50, alpha=0.6, color="#2ECC71", label=f"Repaid   (n={len(repaid):,})",    density=True)
ax.hist(defaulted, bins=50, alpha=0.6, color="#E74C3C", label=f"Defaulted (n={len(defaulted):,})", density=True)
ax.axvline(repaid.mean(),    color="#1E8449", linestyle="--", linewidth=2,
           label=f"Repaid Avg: ${repaid.mean():,.0f}")
ax.axvline(defaulted.mean(), color="#922B21", linestyle="--", linewidth=2,
           label=f"Defaulted Avg: ${defaulted.mean():,.0f}")
ax.set_title("Income Distribution: Defaulters vs Non-Defaulters", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Annual Income (USD)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.legend(fontsize=10)
ax.set_xlim(0, 200000)
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_07_income_distribution.png", bbox_inches="tight")
plt.show()
print(f"Avg Income — Repaid   : ${repaid.mean():,.0f}")
print(f"Avg Income — Defaulted: ${defaulted.mean():,.0f}")
print(f"Income Gap            : ${repaid.mean()-defaulted.mean():,.0f}")


## Chart 4 — DTI Bracket vs Default Rate
**Business Question**: Does higher debt burden mean more defaults?

In [ ]:

order = ["Low DTI","Moderate DTI","High DTI","Very High DTI"]
dti_df = df.groupby("dti_bracket").agg(
    total=("default_flag","count"),
    defaults=("default_flag","sum")
).reindex(order).reset_index()
dti_df["default_rate"] = dti_df["defaults"] / dti_df["total"] * 100

colors = ["#1A5276","#2E86C1","#E67E22","#C0392B"]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(dti_df["dti_bracket"], dti_df["default_rate"], color=colors, edgecolor="white", width=0.55)
ax.set_title("Default Rate by DTI Bracket (Debt-to-Income Proxy)", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("DTI Bracket", fontsize=11); ax.set_ylabel("Default Rate (%)", fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, dti_df["default_rate"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f"{val:.1f}%", ha="center", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_08_dti_default.png", bbox_inches="tight")
plt.show()


## Chart 5 — Credit History Length vs Default Rate
**Business Question**: Does credit seniority reduce default risk?

In [ ]:

df["cred_hist_band"] = pd.cut(df["cb_person_cred_hist_length"],
    bins=[0,3,7,15,50], labels=["1-3 yrs","4-7 yrs","8-15 yrs","15+ yrs"])

ch_df = df.groupby("cred_hist_band", observed=True).agg(
    total=("default_flag","count"), defaults=("default_flag","sum")).reset_index()
ch_df["default_rate"] = ch_df["defaults"] / ch_df["total"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ch_df["cred_hist_band"].astype(str), ch_df["default_rate"],
        marker="o", markersize=9, linewidth=2.5, color="#2E86C1")
ax.fill_between(range(len(ch_df)), ch_df["default_rate"], alpha=0.15, color="#2E86C1")
for i, (x, y) in enumerate(zip(ch_df["cred_hist_band"].astype(str), ch_df["default_rate"])):
    ax.annotate(f"{y:.1f}%", (i, y), textcoords="offset points",
                xytext=(0, 10), ha="center", fontweight="bold")
ax.set_title("Default Rate by Credit History Length", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Credit History Band", fontsize=11); ax.set_ylabel("Default Rate (%)", fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_09_credit_history.png", bbox_inches="tight")
plt.show()


## Chart 6 — Prior Default History vs Current Default Rate
**Business Question**: Are repeat defaulters more likely to fail again?

In [ ]:

prior_df = df.groupby("cb_person_default_on_file").agg(
    total=("default_flag","count"), defaults=("default_flag","sum")).reset_index()
prior_df["default_rate"] = prior_df["defaults"] / prior_df["total"] * 100
prior_df["label"] = prior_df["cb_person_default_on_file"].map({"N":"No Prior Default","Y":"Prior Default on File"})

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(prior_df["label"], prior_df["default_rate"],
              color=["#2ECC71","#E74C3C"], edgecolor="white", width=0.4)
ax.set_title("Default Rate: Prior vs No Prior Default (Credit Bureau)", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("Default Rate (%)", fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, prior_df["default_rate"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f"{val:.1f}%", ha="center", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_10_prior_default.png", bbox_inches="tight")
plt.show()
print(prior_df[["label","total","defaults","default_rate"]].to_string(index=False))


## Chart 7 — Correlation Heatmap
**Business Question**: Which numeric features are most related to default?

In [ ]:

num_cols = ["person_age","person_income","person_emp_length",
            "loan_amnt","loan_int_rate","loan_percent_income",
            "cb_person_cred_hist_length","default_flag"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlBu_r",
            center=0, square=True, ax=ax, linewidths=0.5,
            cbar_kws={"shrink":0.8})
ax.set_title("Feature Correlation Heatmap\n(Focus: correlation with default_flag)",
             fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_11_correlation_heatmap.png", bbox_inches="tight")
plt.show()
print("\nCorrelation with default_flag:")
print(corr["default_flag"].drop("default_flag").sort_values(ascending=False).round(3).to_string())


## Chart 8 — Home Ownership vs Default Rate
**Business Question**: Does housing stability affect repayment?

In [ ]:

own_df = df.groupby("person_home_ownership").agg(
    total=("default_flag","count"), defaults=("default_flag","sum")).reset_index()
own_df["default_rate"] = own_df["defaults"] / own_df["total"] * 100
own_df = own_df.sort_values("default_rate", ascending=False)

colors = ["#C0392B" if r > 25 else "#E67E22" if r > 18 else "#2ECC71"
          for r in own_df["default_rate"]]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(own_df["person_home_ownership"], own_df["default_rate"],
              color=colors, edgecolor="white", width=0.5)
ax.set_title("Default Rate by Home Ownership Type", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Home Ownership"); ax.set_ylabel("Default Rate (%)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, own_df["default_rate"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f"{val:.1f}%", ha="center", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_12_home_ownership.png", bbox_inches="tight")
plt.show()
